In [ ]:
# Detection Image

import cv2 
import mediapipe as mp

# Setup
mp_pose = mp.solutions.pose 
mp_drawing = mp.solutions.drawing_utils

# Read image
image = cv2.imread(r'c:\Users\User\Downloads\images (2).jfif')
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Procss
with mp_pose.Pose(static_image_mode = True) as pose: #Select static to image
    results = pose.process(image_rgb)

    # Draw landmark
    if results.pose_landmarks:
        mp_drawing.draw_landmarks( 
            image,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

        # print landmarks positions
        for idx, landmark in enumerate(results.pose_landmarks.landmark):
            print(f"landmarks {idx}: x={landmark.x:.3f}, z={landmark.z:.3f}")

#Display
cv2.imshow('Pose Detection', image)
cv2.waitKey(0)
cv2.destroyAllWindows()

landmarks 0: x=0.417, z=-0.650
landmarks 1: x=0.447, z=-0.626
landmarks 2: x=0.465, z=-0.626
landmarks 3: x=0.482, z=-0.626
landmarks 4: x=0.400, z=-0.602
landmarks 5: x=0.386, z=-0.602
landmarks 6: x=0.375, z=-0.602
landmarks 7: x=0.512, z=-0.416
landmarks 8: x=0.369, z=-0.305
landmarks 9: x=0.452, z=-0.568
landmarks 10: x=0.396, z=-0.536
landmarks 11: x=0.664, z=-0.278
landmarks 12: x=0.261, z=-0.113
landmarks 13: x=0.903, z=-0.165
landmarks 14: x=0.097, z=0.142
landmarks 15: x=0.737, z=-0.363
landmarks 16: x=0.222, z=0.133
landmarks 17: x=0.690, z=-0.439
landmarks 18: x=0.244, z=0.122
landmarks 19: x=0.660, z=-0.432
landmarks 20: x=0.281, z=0.105
landmarks 21: x=0.664, z=-0.349
landmarks 22: x=0.281, z=0.137
landmarks 23: x=0.591, z=-0.031
landmarks 24: x=0.355, z=0.031
landmarks 25: x=0.567, z=-0.191
landmarks 26: x=0.359, z=0.069
landmarks 27: x=0.571, z=0.583
landmarks 28: x=0.366, z=0.680
landmarks 29: x=0.576, z=0.648
landmarks 30: x=0.369, z=0.735
landmarks 31: x=0.548, z=0.36

In [1]:
# Detection from webcam

import cv2 
import mediapipe as mp

# Setup
mp_pose = mp.solutions.pose 
mp_drawing = mp.solutions.drawing_utils

# Read video
cap = cv2.VideoCapture(0)

# Procss
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose: # Open Model
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            continue
        
        frame = cv2.flip(frame, 1)
        
        #Process
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Draw landmark
        if results.pose_landmarks:
            mp_drawing.draw_landmarks( 
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

        cv2.imshow('Video Pose Detection', image) #Display

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()          # ปิดไฟล์วิดีโอ/ปิดกล้องเว็บแคมเพื่อคืนแรมให้คอม
cv2.destroyAllWindows() # บังคับปิดหน้าต่างโชว์ภาพทั้งหมดที่ OpenCV สร้างขึ้นมา

In [1]:
# Miniproject ยกดัมเบล

import cv2
import mediapipe as mp
import numpy as np
import math

# Setup MediaPipe
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# ฟังก์ชันคำนวณมุมระหว่าง 3 จุด (ไหล่, ข้อศอก, ข้อมือ)
def calculate_angle(a, b, c):
    a = np.array(a) # ไหล่ [x, y]
    b = np.array(b) # ข้อศอก [x, y]
    c = np.array(c) # ข้อมือ [x, y]
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle > 180.0:
        angle = 360-angle
        
    return angle

# ตัวแปรสำหรับนับจำนวนครั้ง
counter = 0
stage = None # สถานะแขน: 'down' (เหยียด) หรือ 'up' (งอ)

# เรียกใช้กล้องเว็บแคม (เปลี่ยนเป็นเลข 0 เพื่อเปิดกล้องตัวเอง หรือใส่พาธไฟล์วิดีโอก็ได้)
cap = cv2.VideoCapture(0) 

with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # พลิกหน้าจอให้เหมือนกระจกเงา (จะได้ไม่งงเวลาขยับซ้ายขวา)
        frame = cv2.flip(frame, 1)
        
        # แปลงสีภาพเป็น RGB ให้ MediaPipe ประมวลผล
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        try:
            landmarks = results.pose_landmarks.landmark
            
            # ดึงพิกัดของแขนขวา (ไหล่, ข้อศอก, ข้อมือ)
            shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,
                        landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,
                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
            
            # คำนวณมุมข้อศอก
            angle = calculate_angle(shoulder, elbow, wrist)
            
            # แสดงตัวเลขมุมตรงข้อศอกซ้าย/ขวาในจอ
            # แปลงพิกัดจาก 0-1 ให้เป็นพิกัดพิกเซลจริงบนจอคอม
            h, w, _ = image.shape
            elbow_pos = tuple(np.multiply(elbow, [w, h]).astype(int))
            cv2.putText(image, f"{int(angle)} deg", elbow_pos, 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
            
            # ลอจิกการนับจำนวนครั้ง (Bicep Curl Logic)
            if angle > 160: # แขนเหยียดตรงสุด
                stage = "down"
            if angle < 40 and stage == 'down': # แขนงอสุด และก่อนหน้านี้เหยียดอยู่
                stage = "up"
                counter += 1
                print(f"Count: {counter}")
                       
        except:
            pass
            
        # วาดเส้นและจุด Landmark ลงบนภาพ
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # วาดกล่องข้อความแสดงคะแนนจำนวนครั้งไว้ที่มุมซ้ายบน
        cv2.rectangle(image, (0,0), (250, 80), (0,0,0), -1)
        cv2.putText(image, 'STAGE', (15,25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
        cv2.putText(image, stage, (15,65), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2, cv2.LINE_AA)
        cv2.putText(image, 'REPS', (150,25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), (150,65), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,0,255), 2, cv2.LINE_AA)
        
        # แสดงหน้าต่างผลลัพธ์
        cv2.imshow('Fitness AI Counter', image)

        # กด 'q' เพื่อปิดโปรแกรม
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


Count: 1
Count: 2
Count: 3
Count: 4
Count: 5
Count: 6
Count: 7
Count: 8
Count: 9


In [2]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.6, min_tracking_confidence=0.6)

actions = ['JAB', 'CROSS', 'HOOK', 'UPPERCUT']
# ชี้ตรงเข้าโฟลเดอร์ Video ในไดรฟ์ D ของน้อง (ปรับเป็นตัวพิมพ์ใหญ่ตามรูป)
video_base_dir = "Video/" 

dataset_rows = []
sequence_length = 30 

print("=== เริ่มตัดสับคลิปวิดีโอเป็นตารางตัวเลข ===")

for action_idx, action in enumerate(actions):
    # ค้นหาไฟล์วิดีโอตรงๆ เช่น Video/JAB.mp4
    video_path = os.path.join(video_base_dir, f"{action}.mp4")
    
    if not os.path.exists(video_path):
        print(f"⚠️ ไม่พบไฟล์วิดีโอ: {video_path} (ข้าม)")
        continue
        
    cap = cv2.VideoCapture(video_path)
    window_frames = [] 
    
    print(f"🎬 กำลังสกัดพิกัดท่า [{action}] จากไฟล์ -> {action}.mp4")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
            
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            # ดึงท่อนบน 7 จุด: จมูก(0), ไหล่ซ้าย(11), ไหล่ขวา(12), ศอกซ้าย(13), ศอกขวา(14), ข้อมือซ้าย(15), ข้อมือขวา(16)
            current_frame_features = []
            for joint in [0, 11, 12, 13, 14, 15, 16]:
                current_frame_features.extend([
                    landmarks[joint].x, 
                    landmarks[joint].y, 
                    landmarks[joint].z, 
                    landmarks[joint].visibility
                ])
            
            window_frames.append(current_frame_features)
            
            if len(window_frames) == sequence_length:
                flatten_row = list(np.array(window_frames).flatten())
                flatten_row.append(action_idx) 
                dataset_rows.append(flatten_row)
                window_frames.pop(0) 
                
    cap.release()

if len(dataset_rows) > 0:
    df = pd.DataFrame(dataset_rows)
    df.rename(columns={df.columns[-1]: 'label'}, inplace=True)
    df.to_csv("boxing_dataset.csv", index=False)
    print(f"\n✨ [สำเร็จ!] เจนไฟล์ตารางออกมาแล้ว -> boxing_dataset.csv (ได้ข้อมูลทั้งหมด {len(df)} แถว)")
else:
    print("\n❌ เกิดข้อผิดพลาด: ดึงข้อมูลไม่ได้เลย ลองเช็คสเปลลิ่งชื่อไฟล์อีกทีครับ")

AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf

# 1. โหลดโมเดลสมองกลมวยไทยที่เราเทรนเสร็จมาใช้งาน
model = tf.keras.models.load_model("muaythai_lstm_model.h5")

# 2. ตั้งค่า MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

# รายชื่อท่าตามลำดับ Label (0=JAB, 1=CROSS, 2=HOOK, 3=UPPERCUT)
actions = ['JAB', 'CROSS', 'HOOK', 'UPPERCUT']

# ตัวแปรสำหรับทำ Sliding Window (เก็บ 30 เฟรมล่าสุด)
sequence_data = []
sequence_length = 30
current_action = "Waiting..."
confidence = 0.0

# 3. เริ่มเปิดกล้องเว็บแคม (เลข 0 คือกล้องหลักของคอมพิวเตอร์)
cap = cv2.VideoCapture(0)
print("=== เปิดกล้องเว็บแคมสำเร็จ! กด 'q' บนคีย์บอร์ดเพื่อปิดหน้าจอ ===")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
        
    # พลิกภาพซ้ายขวาให้เหมือนกระจกเงา จะได้ออกหมัดตามได้ง่าย
    frame = cv2.flip(frame, 1)
    
    # แปลงสีภาพให้ MediaPipe นำไปประมวลผล
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    if results.pose_landmarks:
        # วาดเส้นโครงกระดูกบนหน้าจอโชว์ความเท่
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        landmarks = results.pose_landmarks.landmark
        
        # สกัดพิกัด 7 จุดหลักท่อนบน (สูตรครึ่งตัวแถมจมูก) เหมือนตอนทำ Dataset
        current_frame_features = []
        for joint in [0, 11, 12, 13, 14, 15, 16]:
            current_frame_features.extend([
                landmarks[joint].x, 
                landmarks[joint].y, 
                landmarks[joint].z, 
                landmarks[joint].visibility
            ])
            
        # สะสมฟีเจอร์ลงในลิสต์ชุดลำดับเวลา
        sequence_data.append(current_frame_features)
        
        # ล็อกความยาวไว้ที่ 30 เฟรมล่าสุดเสมอ (ถ้าเกินให้เตะเฟรมเก่าสุดทิ้ง)
        if len(sequence_data) > sequence_length:
            sequence_data.pop(0)
            
        # เมื่อข้อมูลสะสมครบ 30 เฟรม ยัดเข้าปากโมเดลเพื่อ Predict ทันที
        if len(sequence_data) == sequence_length:
            # ปรับมิติข้อมูลให้ตรงกับที่โมเดลต้องการ -> (1, 30, 28)
            input_data = np.expand_dims(sequence_data, axis=0)
            
            # สั่ง Predict พ่นค่าความน่าจะเป็นออกมา
            result = model.predict(input_data, verbose=0)[0]
            
            # หาตำแหน่งคลาสที่มีค่าเปอร์เซ็นต์ความมั่นใจสูงที่สุด
            max_idx = np.argmax(result)
            confidence = result[max_idx]
            
            # ถ้า AI มั่นใจเกิน 75% ให้แสดงชื่อท่าทางนั้นออกมา
            if confidence > 0.75:
                current_action = actions[max_idx]
            else:
                current_action = "Analyzing..."

    # ==========================================
    # 4. วาดแถบ UI ด้านบนหน้าจอเพื่อแสดงผล
    # ==========================================
    # วาดกล่องสี่เหลี่ยมสีดำเป็นพื้นหลังแถบข้อความ
    cv2.rectangle(image, (0, 0), (640, 60), (45, 45, 45), -1)
    
    # พิมพ์ชื่อท่ามวยที่ AI ตรวจจับได้ออกมา
    cv2.putText(image, f"ACTION: {current_action}", (20, 42), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0) if current_action in actions else (255, 255, 255), 3)
    
    # พิมพ์ค่าเปอร์เซ็นต์ความแม่นยำ (Confidence)
    cv2.putText(image, f"CONF: {confidence*100:.1f}%", (400, 42), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 200, 0), 2)
    
    # เปิดหน้าต่างแสดงผล
    cv2.imshow('MuayVerse - Real-time LSTM Test', image)
    
    # ถ้ากดปุ่ม 'q' บนคีย์บอร์ด จะทำการปิดหน้าต่างกล้อง
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

ImportError: cannot import name 'runtime_version' from 'google.protobuf' (d:\Muayverse_Remake\.venv\Lib\site-packages\google\protobuf\__init__.py)